# Strands Decider on FCC complaints

This notebook evaluates the local CPU-served Strands Decider 2B model on the prepared FCC slice. Start `download_deploy_strands_decider.py` first, then run `download_prepare_fcc.py`.

In [ ]:
from pathlib import Path
import json
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, ConfusionMatrixDisplay

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'decision-model-eval' and (PROJECT_DIR / 'decision-model-eval').exists():
    PROJECT_DIR = PROJECT_DIR / 'decision-model-eval'
DATA_PATH = PROJECT_DIR / 'data' / 'fcc_latest_500_prepared.jsonl'
RESULTS_DIR = PROJECT_DIR / 'results'
ENDPOINT = 'http://127.0.0.1:8000'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists(), f'Missing {DATA_PATH}; run download_prepare_fcc.py first.'

In [ ]:
health = requests.get(f'{ENDPOINT}/health', timeout=20)
health.raise_for_status()
health.json()

In [ ]:
records = [json.loads(line) for line in DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
manifest_path = DATA_PATH.with_name('fcc_latest_500_manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
label_counts = pd.Series([record['gold_label'] for record in records]).value_counts()
display(label_counts.rename_axis('gold_label').reset_index(name='records'))
len(records), records[0].keys()

## Run zero-shot routing

The request contains one typed choice question per complaint. The raw response is retained so parser or model changes can be audited.

In [ ]:
def run_one(record):
    payload = {
        'state': record['state'],
        'questions': {
            'queue': {
                'type': 'choice',
                'instructions': record['question'],
                'criteria': record['criteria'],
            }
        },
    }
    started = time.perf_counter()
    try:
        response = requests.post(f'{ENDPOINT}/v1/systemone', json=payload, timeout=120)
        elapsed_ms = (time.perf_counter() - started) * 1000
        response.raise_for_status()
        body = response.json()
        answer = body.get('answers', {}).get('queue', {})
        probabilities = answer.get('probabilities', {})
        prediction = answer.get('choice') or (max(probabilities, key=probabilities.get) if probabilities else None)
        if prediction not in record['options']:
            prediction = None
        return {
            'record_id': record['record_id'],
            'gold_label': record['gold_label'],
            'prediction': prediction,
            'probabilities': probabilities,
            'latency_ms': elapsed_ms,
            'status': 'ok' if prediction else 'invalid_response',
            'raw_response': body,
        }
    except Exception as exc:
        return {
            'record_id': record['record_id'],
            'gold_label': record['gold_label'],
            'prediction': None,
            'probabilities': {},
            'latency_ms': (time.perf_counter() - started) * 1000,
            'status': 'error',
            'error': repr(exc),
        }

results = [run_one(record) for record in records]
results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
valid = results_df[results_df['prediction'].notna()].copy()
labels = sorted(set(results_df['gold_label']))
summary = {
    'model_health': health.json(),
    'dataset_manifest': manifest,
    'records': len(results_df),
    'valid_predictions': len(valid),
    'invalid_or_failed': int(len(results_df) - len(valid)),
    'accuracy': accuracy_score(valid['gold_label'], valid['prediction']) if len(valid) else None,
    'macro_f1': f1_score(valid['gold_label'], valid['prediction'], labels=labels, average='macro', zero_division=0) if len(valid) else None,
    'mean_latency_ms': float(results_df['latency_ms'].mean()),
}
summary

In [ ]:
if len(valid):
    print(classification_report(valid['gold_label'], valid['prediction'], labels=labels, zero_division=0))
    display(pd.DataFrame(confusion_matrix(valid['gold_label'], valid['prediction'], labels=labels), index=labels, columns=labels))
    ConfusionMatrixDisplay.from_predictions(valid['gold_label'], valid['prediction'], labels=labels, xticks_rotation=45, cmap='Blues')
    plt.tight_layout()
    plt.show()

In [ ]:
run_id = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
results_df.to_json(RESULTS_DIR / f'strands_fcc_{run_id}.jsonl', orient='records', lines=True)
(RESULTS_DIR / f'strands_fcc_{run_id}_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Saved results under {RESULTS_DIR}')